# Connecting the trailer bills with the original budget bill
This file provides to connect the trailer bills with the original budget bill.

In [2]:
import pandas as pd

## 1. Consolidate all trailer bill files

In [11]:
# 1. The list of files to read, with labels for the new column
files = [
    {"name": "2023_AB102_CA_Budget_money_only.csv", "label": "AB102"},
    {"name": "2023_SB104_CA_Budget_money_only.csv", "label": "SB104"},
    {"name": "2023_SB105_CA_Budget_money_only.csv", "label": "SB105"}
]

all_dfs = []

for f in files:
    try:
        # Read each CSV file into a DataFrame
        df = pd.read_csv(f["name"])
        
        # Add a new column for the source of the trailer bill
        df.insert(0, "trailer_bill_source", f["label"])
        
        all_dfs.append(df)
        print(f"{f['label']} is read ({len(df)} rows)")
    except Exception as e:
        print(f"Error: Failed to read {f['label']}: {e}")

# 2. concatenate all DataFrames into one
combined_df = pd.concat(all_dfs, ignore_index=True)

# 3. delete rows where "budget_year" is 2022 (adjust the column name if it's different)
combined_df = combined_df[combined_df["budget_year"] != 2022]
combined_df = combined_df[combined_df["budget_year"] != "2022"]

# 4. Save the combined DataFrame to a new Excel file
print(f"\nConsolidated. Total rows: {len(combined_df)}")
combined_df.to_csv("combined_trailer_bills.csv", index=False)
print("Combined file saved as 'combined_trailer_bills.csv'.")

# Check the first few rows of the combined DataFrame
combined_df.head()

AB102 is read (682 rows)
SB104 is read (278 rows)
SB105 is read (26 rows)

Consolidated. Total rows: 981
Combined file saved as 'combined_trailer_bills.csv'.


,trailer_bill_source,level,budget_year,action_type,ref_source,agency,item_number,program_code,object_code,fund_code,fund_name,schedule_group,name,amount
0,AB102,item,2023,Amended,Original,NaN,0250-001-0001,NaN,NaN,1,NaN,NaN,For support of Judicial Branch,620021000
1,AB102,item,2023,Amended,Original,NaN,0250-001-0001,130.0,NaN,1,NaN,1.0,Supreme Court,55790000
2,AB102,item,2023,Amended,Original,NaN,0250-001-0001,135.0,NaN,1,NaN,2.0,Courts of Appeal,271488000
3,AB102,item,2023,Amended,Original,NaN,0250-001-0001,140.0,NaN,1,NaN,3.0,Judicial Council,286639000
4,AB102,item,2023,Amended,Original,NaN,0250-001-0001,155.0,NaN,1,NaN,4.0,Habeas Corpus Resource Center,18597000


## 2. Connecting this csv file with the file of the original budget bill
### 2.1. Only use "Amended" articles.

In [8]:
# 1. Load the original budget data and the combined trailer bills data
df_original = pd.read_csv("CA_Budget_SB101_2023_money_only.csv")
df_trailer = pd.read_csv("combined_trailer_bills.csv")

# 2. Define the keys for merging
keys = ["item_number", "program_code", "object_code"]

# 3. Merge the original data with the trailer bill data on the keys
df_merged = pd.merge(
    df_original, 
    df_trailer[keys + ["amount", "trailer_bill_source"]], 
    on=keys, 
    how="left", 
    suffixes=("", "_revised")
)

# 4. Set the new columns "revised_amount" and "trailer_bill_source"
# If there is a revised amount, use it; otherwise, keep the original amount
df_merged["revised_amount"] = df_merged["amount_revised"].fillna(df_merged["amount"])

# If there is a trailer bill source, use it; otherwise, set it to "original"
df_merged["trailer_bill_source"] = df_merged["trailer_bill_source"].fillna("original")

# 5. Drop the intermediate columns used for merging
df_merged = df_merged.drop(columns=["amount_revised"])

# 6. Save the merged DataFrame to a new CSV file
df_merged.to_csv("2023_SB101_with_revisions.csv", index=False, encoding='utf-8-sig')

print("Complete merged file saved as '2023_SB101_with_revisions.csv'.")
print(df_merged["trailer_bill_source"].value_counts()) # Check the count of each source type

Complete merged file saved as '2023_SB101_with_revisions.csv'.
trailer_bill_source
original    3792
AB102        574
SB104        218
SB105          8
Name: count, dtype: int64


## 2.2 Add "Added" articles

In [10]:
# 1. Load the files
dtype_spec = {"item_number": str, "program_code": str, "object_code": str}
df_revisions = pd.read_csv("2023_SB101_with_revisions.csv", dtype=dtype_spec)
df_trailer = pd.read_csv("combined_trailer_bills.csv", dtype=dtype_spec)
keys = ["item_number", "program_code", "object_code"]

# 2. Extract rows from df_trailer that are not in df_revisions using a merge-filter approach
merged_check = pd.merge(df_trailer, df_revisions[keys], on=keys, how="left", indicator=True)
df_new_entries = merged_check[merged_check["_merge"] == "left_only"].drop(columns=["_merge"]).copy()

# 3. Align columns and data
df_new_entries["revised_amount"] = df_new_entries["amount"]
df_new_entries["amount"] = 0  

for col in df_revisions.columns:
    if col not in df_new_entries.columns:
        df_new_entries[col] = ""

# 4. Append and ensure column order matches
df_new_entries = df_new_entries[df_revisions.columns]
df_final = pd.concat([df_revisions, df_new_entries], ignore_index=True)

# 5. Save
df_final.to_csv("2023_SB101_final_comprehensive.csv", index=False, encoding='utf-8-sig')

print(f"The number of rows in the revised file: {len(df_revisions)}")
print(f"The number of new added rows: {len(df_new_entries)}")
print(f"The number of rows in the final comprehensive file: {len(df_final)}")

The number of rows in the revised file: 4592
The number of new added rows: 189
The number of rows in the final comprehensive file: 4781
